# **Notebook 1: Model Training and Validation (Proof of Concept)**

>### **Project: PestNeuroVision**
>
>**Description:** This notebook describes the complete process of defining the architecture and training the PestNeuroVision model.
>
>**Important note about the dataset:** For this public repository and to demonstrate reproducibility, this notebook uses a selected subset of data (miniset data). This subset consists exclusively of CC0 licensed images to ensure full compliance with open access standards and intellectual property rights. The complete dataset used in the original research is not included because several photographs are subject to restrictive copyrights that prevent their public redistribution.
>
>The main objective of this notebook is to demonstrate the technical feasibility and functional integrity of the training workflow. While the complete study results presented in the article may involve a larger, private, or restricted dataset, this script provides the exact environment and logic used during the research.
>
>
>---
>
>**Output:** A file named `best.pt` which stores the trained weights. This will be used in various snotebooks for further analysis.
>
>---
>
>**License:** This notebook is licensed under the GNU Affero General Public License v3.0 (AGPL-3.0).
See details in the [GitHub repository](https://github.com/alexander-lm/PestNeuroVision/tree/main).

###**System and CPU information**

In [ ]:
# System information
!cat /etc/os-release

In [ ]:
#GPU information
!nvidia-smi

###**Download the mini dataset (images) for model training and validation**

In [ ]:
# Download the mini dataset (images) for model training and validation
# Due to copyright policies, only images in the public domain (CC0 license) are available.
# If you wish to train the model with more data, you can obtain images directly from the iNaturalist, Roboflow, and Kaggle repositories.

!git init PestNeuroVision
%cd PestNeuroVision
!git remote add origin https://github.com/alexander-lm/PestNeuroVision.git
!git sparse-checkout init --cone
!git sparse-checkout set "dataset/mini_training_dataset_(CC0 1.0)"
!git pull --depth 1 origin main
!mv "dataset/mini_training_dataset_(CC0 1.0)" /content/training_minidataset
%cd /content
!rm -rf PestNeuroVision

###**Configuration file for model training**

In [ ]:
import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  data = {
      'path': '/content/training_minidataset',
      'train': 'train/images',
      'val': 'validation/images',
      'test': 'test/images',
      'nc': number_of_classes,
      'names': classes
  }

  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

path_to_classes_txt = '/content/training_minidataset/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat ./data.yaml

###**YOLO11s model training**

In [ ]:
!pip install Ultralytics

In [ ]:
import random
import numpy as np
import torch
import matplotlib
from ultralytics import settings, YOLO

In [ ]:
matplotlib.rcParams['savefig.dpi'] = 500
matplotlib.rcParams['figure.autolayout'] = True

SEED = 50
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

settings.update({"tensorboard": True})

model = YOLO("yolo11s.pt")
results = model.train(
    data="./data.yaml",
    epochs=200,
    imgsz=640,
    seed=SEED,
    deterministic=True
)

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True,

/usr/local/lib/python3.12/dist-packages/torch/jit/_trace.py:1016: UserWarning: The input to trace is already a ScriptModule, tracing it is a no-op. Returning the object as is.
  traced_func = _trace_impl(


      1/200      4.14G      2.005      4.723      1.868         11        640: 100% ━━━━━━━━━━━━ 4/4 5.5s/it 21.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.9s/it 3.9s
                   all         13         31      0.524     0.0842     0.0372     0.0172

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/200      4.18G       1.66      4.215      1.803          9        640: 100% ━━━━━━━━━━━━ 4/4 2.5it/s 1.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.6it/s 0.2s
                   all         13         31      0.268      0.222      0.124     0.0757

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/200      4.18G      1.351       3.54      1.593         10        640: 100% ━━━━━━━━━━━━ 4/4 3.3it/s 1.2s
                 Class     Images  Instances      Box(P          R     